In [52]:
import seaborn as sns
import pandas as pd
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.tree import export_text
from sklearn import model_selection
from sklearn import preprocessing
from sklearn import metrics
import tensorflow as tf

In [38]:
ctrain=pd.read_csv('train.csv',index_col=0)
train=pd.DataFrame(ctrain)
ctest=pd.read_csv('test.csv',index_col=0)
test=pd.DataFrame(ctest)
train.head()

,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
PassengerId,,,,,,,,,,,
1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [39]:
train.isnull().sum()

,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,177
SibSp,0
Parch,0
Ticket,0
Fare,0
Cabin,687


In [40]:
test.isnull().sum()

,0
Pclass,0
Name,0
Sex,0
Age,86
SibSp,0
Parch,0
Ticket,0
Fare,1
Cabin,327
Embarked,0


In [41]:
train=train.drop(['Name','Cabin','Ticket'],axis=1)
test=test.drop(['Name','Cabin','Ticket'],axis=1)

In [42]:
df=train
Q1 = df['Fare'].quantile(0.25)
Q3 = df['Fare'].quantile(0.75)
IQR = Q3 - Q1

# Define outlier range
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Find outliers
outliers = df[(df['Fare'] < lower_bound) | (df['Fare'] > upper_bound)]

biggest_outlier = outliers['Fare'].max()
smallest_outlier = outliers['Fare'].min()
print("Biggest outlier:", biggest_outlier, smallest_outlier)

smallest=df['Fare'].min()
print(smallest)

Biggest outlier: 512.3292 66.6
0.0


In [43]:
train = train[train['Fare'] < 500]

In [44]:
train['Sex'] = train['Sex'].map( {'female': 1, 'male': 0} ).astype(int)
test['Sex'] = test['Sex'].map( {'female': 1, 'male': 0} ).astype(int)

freq_port = train.Embarked.dropna().mode()[0]
#for dataset in combine:
train['Embarked'] = train['Embarked'].fillna(freq_port)
test['Fare'].fillna(test['Fare'].dropna().median(), inplace=True)


train.isnull().sum()

<ipython-input-44-8a5d1344a599>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train['Sex'] = train['Sex'].map( {'female': 1, 'male': 0} ).astype(int)
<ipython-input-44-8a5d1344a599>:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train['Embarked'] = train['Embarked'].fillna(freq_port)
<ipython-input-44-8a5d1344a599>:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace meth

,0
Survived,0
Pclass,0
Sex,0
Age,177
SibSp,0
Parch,0
Fare,0
Embarked,0


In [45]:
train['Embarked'] = train['Embarked'].map( {'Q':2,'S': 1, 'C': 0} ).astype(int)
test['Embarked'] = test['Embarked'].map( {'Q':2,'S': 1, 'C': 0} ).astype(int)
train.head()

<ipython-input-45-9fbb842f572a>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train['Embarked'] = train['Embarked'].map( {'Q':2,'S': 1, 'C': 0} ).astype(int)


,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
PassengerId,,,,,,,,
1,0,3,0,22.0,1,0,7.2500,1
2,1,1,1,38.0,1,0,71.2833,0
3,1,3,1,26.0,0,0,7.9250,1
4,1,1,1,35.0,1,0,53.1000,1
5,0,3,0,35.0,0,0,8.0500,1


In [46]:
guess_ages = np.zeros((2,3))
guess_ages

array([[0., 0., 0.],
       [0., 0., 0.]])

In [47]:
combine=[train,test]
for dataset in combine:
    for i in range(0, 2):
        for j in range(0, 3):
            guess_df = dataset[(dataset['Sex'] == i) & \
                                  (dataset['Pclass'] == j+1)]['Age'].dropna()

            age_guess = guess_df.median()
            guess_ages[i,j] = age_guess

    for i in range(0, 2):
        for j in range(0, 3):
            dataset.loc[(dataset['Age'].isnull()) & (dataset['Sex'] == i) & (dataset['Pclass'] == j+1),'Age'] = guess_ages[i,j]

    dataset['Age'] = dataset['Age'].astype(int)

train.head()

<ipython-input-47-1563827f58ec>:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataset['Age'] = dataset['Age'].astype(int)


,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
PassengerId,,,,,,,,
1,0,3,0,22,1,0,7.2500,1
2,1,1,1,38,1,0,71.2833,0
3,1,3,1,26,0,0,7.9250,1
4,1,1,1,35,1,0,53.1000,1
5,0,3,0,35,0,0,8.0500,1


In [48]:
combine=[train,test]
for dataset in combine:
    dataset['Family']=dataset['SibSp']+dataset['Parch']
    dataset=dataset.drop(['SibSp','Parch'],axis=1,inplace=True)
train.head()
test.head()

<ipython-input-48-96e6a0e4fafb>:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataset['Family']=dataset['SibSp']+dataset['Parch']
<ipython-input-48-96e6a0e4fafb>:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataset=dataset.drop(['SibSp','Parch'],axis=1,inplace=True)


,Pclass,Sex,Age,Fare,Embarked,Survived,Family
PassengerId,,,,,,,
892,3,0,34,7.8292,2,0,0
893,3,1,47,7.0000,1,1,1
894,2,0,62,9.6875,2,0,0
895,3,0,27,8.6625,1,0,0
896,3,1,22,12.2875,1,1,2


In [49]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

def standardize():
  for column_name in train.columns:
      if column_name != 'Survived' and column_name != 'Sex':
        column_train = train[column_name].to_numpy().reshape(-1,1)
        scaler.fit(column_train)
        # train data
        scaled_train = scaler.transform(column_train)
        train[column_name] = scaled_train
        # test data
        column_test = test[column_name].to_numpy().reshape(-1,1)
        scaled_test = scaler.transform(column_test)
        test[column_name] = scaled_test

standardize()
test.head()

<ipython-input-49-db48385b16dc>:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train[column_name] = scaled_train
<ipython-input-49-db48385b16dc>:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train[column_name] = scaled_train
<ipython-input-49-db48385b16dc>:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/i

,Pclass,Sex,Age,Fare,Embarked,Survived,Family
PassengerId,,,,,,,
892,0.824123,0,0.366325,-0.552885,2.144397,0,-0.561424
893,0.824123,1,1.338030,-0.573034,0.193347,1,0.057886
894,-0.375584,0,2.459227,-0.507729,2.144397,0,-0.561424
895,0.824123,0,-0.156900,-0.532636,0.193347,0,-0.561424
896,0.824123,1,-0.530633,-0.444550,0.193347,1,0.677196


In [50]:
X_train = train.drop('Survived', axis=1)  # Features
y_train = train['Survived']
X_test = test.drop('Survived', axis=1)  # Features
y_test = test['Survived']

In [90]:
# Parametri mreze
learning_rate = 0.001
nb_epochs = 50
batch_size = 64

# Parametri arhitekture
nb_input = len(X_train.columns)
nb_hidden1 = 128  # 1st layer number of neurons
nb_hidden2 = 128  # 2nd layer number of neurons
nb_classes = 2

# Sama mreza
# w = {
#     '1': tf.Variable(tf.random.normal([nb_input, nb_hidden1], dtype=tf.float64)),
#     '2': tf.Variable(tf.random.normal([nb_hidden1, nb_hidden2], dtype=tf.float64)),
#     'out': tf.Variable(tf.random.normal([nb_hidden2, nb_classes], dtype=tf.float64))
# }

# b = {
#     '1': tf.Variable(tf.random.normal([nb_hidden1], dtype=tf.float64)),
#     '2': tf.Variable(tf.random.normal([nb_hidden2], dtype=tf.float64)),
#     'out': tf.Variable(tf.random.normal([nb_classes], dtype=tf.float64))
# }

w = {
    '1': tf.Variable(tf.keras.initializers.GlorotUniform()([nb_input, nb_hidden1], dtype=tf.float64)),
    '2': tf.Variable(tf.keras.initializers.GlorotUniform()([nb_hidden1, nb_hidden2], dtype=tf.float64)),
    'out': tf.Variable(tf.keras.initializers.GlorotUniform()([nb_hidden2, nb_classes], dtype=tf.float64))
}

b = {
    '1': tf.Variable(tf.keras.initializers.GlorotUniform()([nb_hidden1], dtype=tf.float64)),
    '2': tf.Variable(tf.keras.initializers.GlorotUniform()([nb_hidden2], dtype=tf.float64)),
    'out': tf.Variable(tf.keras.initializers.GlorotUniform()([nb_classes], dtype=tf.float64))
}

f = {
    '1': tf.nn.relu,
    '2': tf.nn.relu,
    'out': tf.nn.softmax
}

def runNN(x):
    z1 = tf.add(tf.matmul(x, w['1']), b['1'])
    a1 = f['1'](z1)
    z2 = tf.add(tf.matmul(a1, w['2']), b['2'])
    a2 = f['2'](z2)
    z_out = tf.add(tf.matmul(a2, w['out']), b['out']) # a2 ovde!
    out = f['out'](z_out)

    pred = tf.argmax(out, 1)

    return pred, z_out

# GD je djubre
opt = tf.keras.optimizers.Adam(learning_rate=learning_rate)

# Trening!
for epoch in range(nb_epochs):
    epoch_loss = 0
    nb_batches = int(X_train.shape[0] / batch_size)
    for i in range(nb_batches):
        x = X_train[i*batch_size : (i+1)*batch_size]
        y = y_train[i*batch_size : (i+1)*batch_size]
        y_onehot = tf.one_hot(y, nb_classes)

        with tf.GradientTape() as tape:
            _, z_out = runNN(x)

            loss = tf.reduce_mean(tf.nn.softmax_cross_entropy_with_logits(logits=z_out, labels=y_onehot))

        w1_g, w2_g, wout_g, b1_g, b2_g, bout_g = tape.gradient(loss, [w['1'], w['2'], w['out'], b['1'], b['2'], b['out']])

        opt.apply_gradients(zip([w1_g, w2_g, wout_g, b1_g, b2_g, bout_g], [w['1'], w['2'], w['out'], b['1'], b['2'], b['out']]))

        epoch_loss += loss

    # U svakoj epohi ispisujemo prosečan loss.
    epoch_loss /= len(y_train)

    print(f'Epoch: {epoch+1}/{nb_epochs}| Avg loss: {epoch_loss:.5f}')

# Test!
x = X_test
y = y_test

pred, _ = runNN(X_train)
pred_correct = tf.equal(pred, y_train)
accuracytrain = tf.reduce_mean(tf.cast(pred_correct, tf.float32))

pred, _ = runNN(x)
pred_correct = tf.equal(pred, y)
accuracy = tf.reduce_mean(tf.cast(pred_correct, tf.float32))

print(f'Test acc: {accuracy:.3f}, {accuracytrain}')

Epoch: 1/50| Avg loss: 0.00895
Epoch: 2/50| Avg loss: 0.00745
Epoch: 3/50| Avg loss: 0.00679
Epoch: 4/50| Avg loss: 0.00637
Epoch: 5/50| Avg loss: 0.00616
Epoch: 6/50| Avg loss: 0.00606
Epoch: 7/50| Avg loss: 0.00597
Epoch: 8/50| Avg loss: 0.00590
Epoch: 9/50| Avg loss: 0.00584
Epoch: 10/50| Avg loss: 0.00579
Epoch: 11/50| Avg loss: 0.00575
Epoch: 12/50| Avg loss: 0.00572
Epoch: 13/50| Avg loss: 0.00568
Epoch: 14/50| Avg loss: 0.00565
Epoch: 15/50| Avg loss: 0.00562
Epoch: 16/50| Avg loss: 0.00560
Epoch: 17/50| Avg loss: 0.00557
Epoch: 18/50| Avg loss: 0.00555
Epoch: 19/50| Avg loss: 0.00552
Epoch: 20/50| Avg loss: 0.00550
Epoch: 21/50| Avg loss: 0.00548
Epoch: 22/50| Avg loss: 0.00546
Epoch: 23/50| Avg loss: 0.00543
Epoch: 24/50| Avg loss: 0.00542
Epoch: 25/50| Avg loss: 0.00540
Epoch: 26/50| Avg loss: 0.00538
Epoch: 27/50| Avg loss: 0.00536
Epoch: 28/50| Avg loss: 0.00534
Epoch: 29/50| Avg loss: 0.00532
Epoch: 30/50| Avg loss: 0.00531
Epoch: 31/50| Avg loss: 0.00529
Epoch: 32/50| Avg